# The Docker stack

What runs where: services, profiles, ports and health. These cells only read; start the stack from a terminal.

See the [README](README.md) for the why and the how.

In [ ]:
# Run from anywhere inside the repository: find its root so `marginalia` and the test helpers import,
# and read .env (the same settings the services use).
import os, sys, pathlib
ROOT = next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / 'pyproject.toml').exists())
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print(ROOT)

## The services, by profile
`docker compose config` resolves `docker-compose.yml` with `.env`.

In [ ]:
import json, subprocess
cfg = json.loads(subprocess.run(['docker', 'compose', '--profile', 'app', '--profile', 'airflow', '--profile',
                                 'observability', 'config', '--format', 'json'], capture_output=True, text=True,
                                check=True).stdout)
for name, s in sorted(cfg['services'].items()):
    ports = ', '.join(f"{p.get('host_ip', '')}:{p.get('published')}->{p.get('target')}" for p in s.get('ports', []))
    print(f"{name:22} {','.join(s.get('profiles', ['(always)'])):26} {s.get('image', '(built)'):38} {ports}")

## Volumes: where the data lives

In [ ]:
for name, v in sorted(cfg['volumes'].items()):
    print(f"{name:22} -> {v.get('name')}")

## What is running now (needs Docker)

In [ ]:
print(subprocess.run(['docker', 'compose', 'ps', '--format', 'table {{.Name}}\t{{.Status}}\t{{.Ports}}'],
                     capture_output=True, text=True).stdout)

## Health (needs the stack)
The API answers `/health` (database and Redis) and `/health/live`.

In [ ]:
import httpx
for url in ['http://127.0.0.1:8010/health', 'http://127.0.0.1:8010/health/live', 'http://127.0.0.1:6340/readyz',
            'http://127.0.0.1:11435/api/tags', 'http://127.0.0.1:8090/api/v2/monitor/health']:
    try:
        r = httpx.get(url, timeout=5)
        print(f'{r.status_code} {url} {r.text[:120]}')
    except Exception as e:
        print(f'down {url}: {type(e).__name__}')